<a href="https://colab.research.google.com/github/nehagoyal09/Logistics_Data_Analyst_Intern/blob/main/Week_4_Graph_Plot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Dataset Generation (Logistics Features)
np.random.seed(42)
n = 500

carriers = ['ExpressLog', 'BlueDart_3PL', 'SurfaceFast', 'Delhivery_Express']
hubs = ['FC_North', 'FC_West', 'FC_South', 'FC_East']

df = pd.DataFrame({
    'order_id': [f'ORD-{10000+i}' for i in range(n)],
    'carrier_partner': np.random.choice(carriers, size=n, p=[0.3, 0.25, 0.25, 0.2]),
    'fulfillment_hub': np.random.choice(hubs, size=n),
    'order_weight_kg': np.round(np.random.gamma(shape=2.0, scale=3.5, size=n) + 0.5, 2),
    'distance_km': np.round(np.random.uniform(50, 1500, size=n), 1),
    'cycle_time_hrs': np.round(np.random.normal(loc=45, scale=15, size=n).clip(12, 120), 1)
})

# Freight Cost Formula (Non-linear Target Variable)
df['freight_cost_inr'] = np.round(
    120 + (df['order_weight_kg'] * 32.5) + (df['distance_km'] * 0.45) +
    (df['cycle_time_hrs'] * 2.1) + np.random.normal(0, 20, n), 2
)

# 2. Machine Learning Encoding & Train-Test Split
X = pd.get_dummies(df[['order_weight_kg', 'distance_km', 'cycle_time_hrs', 'carrier_partner', 'fulfillment_hub']], drop_first=True)
y = df['freight_cost_inr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Model Training (Linear Regression vs Random Forest)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# 4. Print Model Performance Metrics
print("--- MODEL PERFORMANCE COMPARISON ---")
print(f"Linear Regression -> MAE: ₹{mean_absolute_error(y_test, y_pred_lr):.2f}, RMSE: ₹{np.sqrt(mean_squared_error(y_test, y_pred_lr)):.2f}, R2: {r2_score(y_test, y_pred_lr)*100:.2f}%")
print(f"Random Forest     -> MAE: ₹{mean_absolute_error(y_test, y_pred_rf):.2f}, RMSE: ₹{np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f}, R2: {r2_score(y_test, y_pred_rf)*100:.2f}%")

# 5. GRAPH 1: Actual vs Predicted Freight Cost (Random Forest)
sns.set_theme(style="whitegrid")
plt.figure(figsize=(6.5, 4))
plt.scatter(y_test, y_pred_rf, alpha=0.6, color='#2B6CB0', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Figure 1: Actual vs. Predicted Freight Shipping Fees (INR) - Random Forest')
plt.xlabel('Actual Freight Cost (INR)')
plt.ylabel('Predicted Freight Cost (INR)')
plt.tight_layout()
plt.show()

# 6. GRAPH 2: Feature Importance Drivers
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(6.5, 4))
feat_imp.plot(kind='barh', color='#1A365D')
plt.title('Figure 2: Random Forest Feature Importance Drivers')
plt.xlabel('Gini Relative Importance Score')
plt.ylabel('Predictor Variable')
plt.tight_layout()
plt.show()